In [1]:
import tensorflow as tf
import sys

sys.path.append('kaggle/input/datasets/harshit1234g/axiomlm-utils')
import llm_components as lc

In [2]:
sp = lc.load_sp_tokenizer('kaggle/input/datasets/harshit1234g/axiomlm-utils/sp_tokenizer.model')

In [3]:
model = tf.keras.models.load_model('kaggle/working/AxiomLM-33M-Base.keras')

In [4]:
def generate_text(
    model,
    tokenizer,
    prompt: str,
    *,
    max_new_tokens: int = 512,
    temperature: float = 1.0,
    top_k: int | None = None,
    eos_id: int | None = None
):
    # Encode prompt
    input_ids = tokenizer.encode(prompt)
    input_ids = tf.constant([input_ids], dtype= tf.int32)

    # Safety: truncate if prompt exceeds context window
    if input_ids.shape[1] > model.seq_len:
        input_ids = input_ids[:, -model.seq_len:]

    # First forward pass (full prompt)
    logits, past = model(
        input_ids,
        past= None,
        use_cache= True,
        training= False
    )

    generated_ids = input_ids.numpy().tolist()[0]

    # Only feed last token from now on
    next_token = input_ids[:, -1:]

    for _ in range(max_new_tokens):

        # Stop if context window limit reached
        total_len = len(generated_ids)
        if total_len >= model.seq_len:
            break

        logits, past = model(
            next_token,
            past= past,
            use_cache= True,
            training= False
        )

        # Take logits of last token
        logits = logits[:, -1, :]

        # Temperature scaling
        if temperature != 1.0:
            logits = logits / temperature

        # Top-k filtering
        if top_k is not None:
            values, _ = tf.math.top_k(logits, k= top_k)
            min_values = values[:, -1, tf.newaxis]
            logits = tf.where(
                logits < min_values,
                tf.fill(tf.shape(logits), -1e10),
                logits
            )

        # Sample
        probs = tf.nn.softmax(logits, axis= -1)
        next_token = tf.random.categorical(tf.math.log(probs), num_samples= 1)

        token_id = int(next_token.numpy()[0][0])
        generated_ids.append(token_id)

        # EOS stopping
        if eos_id is not None and token_id == eos_id:
            break

    return tokenizer.decode(generated_ids)

In [5]:
text = generate_text(
    model,
    sp,
    prompt= 'Robert Boulter',
    eos_id= sp.eos_id(),
    max_new_tokens= 50
)

In [6]:
print(text)

Robert Boulter led an Australian team against Joe Hoakau and his Eastern Australian teammate Hideki O 'Neal . A few moments before dropping , he made 45 of 15 moves into a miss and an appearance against teammates ' Luther " Gruppe " Gro
